Scout site selection strategy

scout locations must first:

1. Be located within the uncovered area.
2. Be within developed or populated land.
3. Have reasonable access to a major road.
4. Be close to potential demand locations.
5. Be at least 3 km from existing stations.

Locations approximately 4–6 km from the nearest existing station will
be preferred. Locations farther away may still be considered when they
have strong demand and road accessibility.

After candidates are ranked, a minimum separation distance will be
applied so that the recommended stations serve different communities
rather than clustering in one area.

In [1]:
from pathlib import Path
import pandas as pd 
import geopandas as gpd
import matplotlib.pyplot as plt 
import osmnx as ox

/Users/manafosman/Documents/my_documents/My_Projects/Data_Science /accra-ev-swap-coverage-analysis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
#defining paths
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

raw_data_directory = project_root/"data"/"raw"
processed_data_directory = project_root/"data"/"Processed"
figures_directory = project_root/"outputs"/"figures"

figures_directory.mkdir(parents=True,exist_ok=True)

In [ ]:
#load data
uncovered_area_file = (processed_data_directory/"preliminary_uncovered_area.gpkg")
station_file = (processed_data_directory/"verified_swap_station_points.geojson")

uncovered_area = gpd.read_file(uncovered_area_file)
stations = gpd.read_file(station_file)


In [4]:
stations_projected = stations.to_crs(uncovered_area.crs)

In [6]:
uncovered_area.crs == stations_projected.crs

True

In [7]:
#download developed land
boundary_file = (raw_data_directory/"greater_accra_boundary.geojson")
study_boundary = gpd.read_file(boundary_file)

In [8]:
study_boundary_wgs84 = study_boundary.to_crs("EPSG:4326")
query_polygon = study_boundary_wgs84.geometry.union_all()

In [14]:
print("Boundary records:", len(study_boundary_wgs84))
print("Boundary CRS:", study_boundary_wgs84.crs)
print("Boundary bounds:", study_boundary_wgs84.total_bounds)
print("Geometry empty:", study_boundary_wgs84.geometry.is_empty.any())
print("Geometry valid:", study_boundary_wgs84.geometry.is_valid.all())

if "display_name" in study_boundary_wgs84.columns:
    print(study_boundary_wgs84["display_name"])

Boundary records: 1
Boundary CRS: EPSG:4326
Boundary bounds: [-0.5197049  5.4706519  0.6722593  6.1076145]
Geometry empty: False
Geometry valid: True
0    Greater Accra Region, Ghana
Name: display_name, dtype: object


In [10]:
developed_land_tags = {"Landuse":["residental","commerical","retail","industrial"]}

In [15]:
ox.settings.use_cache = True
ox.settings.log_console = True

In [17]:
place_name = "Greater Accra Region, Ghana"
developed_land_tags = { "landuse": ["residential","commercial","retail","industrial",]}
developed_features = ox.features_from_place(place_name,tags=developed_land_tags,)
print("Downloaded features:", len(developed_features))

Downloaded features: 1671


In [19]:
developed_land = developed_features[developed_features.geometry.geom_type.isin(["Polygon","Multipolygon"])].copy()

columns_to_keep = [column for column in["landuse","name","geometry"] if column in developed_land.columns]
developed_land = (developed_land[columns_to_keep].reset_index(drop=True))
print("Developed_land polygons:",len(developed_land))

Developed_land polygons: 1659


In [20]:
developed_land_projected =developed_land.to_crs(uncovered_area.crs)
print("match:",developed_land_projected.crs == developed_land_projected.crs)

match: True
